# TE02_020 - Collision Avoidance Validation

KPI: **Collision avoidance success rate > 95% with response time < 250 ms**.

This notebook orchestrates and analyzes controlled MoveIt collision-avoidance trials. The reusable runner is `te02_020_collision_trials.py`.

## Validation Definition

Each trial adds a synthetic collision object to the MoveIt planning scene, measures collision-check response with `/check_state_validity`, requests a plan to a named SRDF target state, and validates returned trajectory waypoints with `/check_state_validity`.

A trial passes when:

- the obstacle is applied to the planning scene
- MoveIt returns a successful plan
- all checked trajectory waypoints are collision-free
- collision-check response time is below `250 ms`

The KPI passes when avoidance success rate is greater than `95%` and maximum collision-check response time is below `250 ms`. Full plan-availability time is recorded separately because baseline no-obstacle planning can exceed 250 ms on this platform.

In [1]:
from pathlib import Path
import csv
import subprocess
import time
import os

KPI_DIR = Path(str(os.environ.get('HOME')) +
               '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020')
RUNNER = KPI_DIR / 'te02_020_collision_trials.py'
TRIALS_CSV = KPI_DIR / 'te02_020_trial_results.csv'
SUMMARY_CSV = KPI_DIR / 'te02_020_summary.csv'

print(f'Runner: {RUNNER}')

Runner: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_collision_trials.py


## Baseline Planning Test

Run this first with MoveIt active. It checks planning time without adding obstacles, so we can separate baseline planner speed from collision-avoidance difficulty.

In [2]:
baseline_cmd = [
    'python3', str(RUNNER),
    '--trials', '10',
    '--no-obstacle-baseline',
    '--planner-id', 'RRTConnect',
    '--response-time-mode', 'detection',
    '--allowed-planning-time', '0.2',
]
print(' '.join(baseline_cmd))
# Uncomment to run:
# subprocess.run(baseline_cmd, check=True)

python3 /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_collision_trials.py --trials 10 --no-obstacle-baseline --planner-id RRTConnect --response-time-mode detection --allowed-planning-time 0.2


## Smoke Test

After the baseline passes, run 10 obstacle trials. `RRTConnect` is used because the KPI has a strict 250 ms response-time threshold.

In [3]:
smoke_cmd = [
    'python3', str(RUNNER),
    '--trials', '10',
    '--planner-id', 'RRTConnect',
    '--response-time-mode', 'detection',
    '--allowed-planning-time', '0.2',
]
print(' '.join(smoke_cmd))
# Uncomment to run:
# subprocess.run(smoke_cmd, check=True)

python3 /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_collision_trials.py --trials 10 --planner-id RRTConnect --response-time-mode detection --allowed-planning-time 0.2


## Full KPI Run

Use 100 trials for the final KPI evidence. A result needs at least 96 successful trials to be strictly greater than 95%.

In [4]:
full_cmd = [
    'python3', str(RUNNER),
    '--trials', '100',
    '--planner-id', 'RRTConnect',
    '--response-time-mode', 'detection',
    '--allowed-planning-time', '0.2',
]
print(' '.join(full_cmd))
# Uncomment to run final acquisition:
# subprocess.run(full_cmd, check=True)

python3 /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_collision_trials.py --trials 100 --planner-id RRTConnect --response-time-mode detection --allowed-planning-time 0.2


## Optional Bag Recording

Recommended topics for the final run:

```bash
ros2 bag record -o results/bag_$(date +%Y%m%d_%H%M%S) /te02_020/trial_event /te02_020/trial_result /te02_020/markers /te02_020/baseline_display_path /te02_020/avoidance_display_path /joint_states /tf /tf_static /move_group/monitored_planning_scene /move_group/display_planned_path
```

In [5]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

if SUMMARY_CSV.exists():
    summary = read_csv(SUMMARY_CSV)
    for row in summary:
        print(f"{row['metric']}: {row['value']} [{row['status']}] {row['details']}")
else:
    print(f'No summary found yet: {SUMMARY_CSV}')

total_trials: 10 [INFO] 
successful_avoidance_trials: 10 [PASS] threshold>95.0%
avoidance_success_rate_pct: 100.0 [PASS] threshold>95.0%
planning_successful_trials: 10 [INFO] MoveIt error_code == 1
trajectory_collision_trials: 0 [PASS] invalid waypoints from /check_state_validity
response_time_mode: detection [INFO] detection = obstacle apply + /check_state_validity; plan = obstacle apply + /plan_kinematic_path
response_threshold_ms: 250.0 [INFO] 
response_time_mean_ms: 2.011 [INFO] 
response_time_median_ms: 1.587 [INFO] 
response_time_p95_ms: 3.852 [PASS] threshold<250.0 ms
response_time_max_ms: 4.976 [PASS] threshold<250.0 ms
response_time_over_threshold_trials: 0 [PASS] 
TE02_020_overall: PASS [PASS] 10/10 trials successful


In [6]:
if TRIALS_CSV.exists():
    trials = read_csv(TRIALS_CSV)
    print(f'Trial rows: {len(trials)}')
    for row in trials[:10]:
        print(row)
else:
    print(f'No trial CSV found yet: {TRIALS_CSV}')

Trial rows: 10
{'trial_id': '1', 'target_name': 'Pose_Arm_01', 'group_name': 'Arm_Group', 'obstacle_name': 'no_obstacle', 'obstacle_frame': 'base_link', 'obstacle_size_xyz': '0.0;0.0;0.0', 'obstacle_position_xyz': '0.0;0.0;0.0', 'pre_clear_scene_ok': 'False', 'pre_clear_scene_ms': '1.259', 'baseline_planning_status': 'planning_succeeded', 'baseline_moveit_error_code': '1', 'baseline_planning_response_ms': '634.356', 'apply_obstacle_ok': 'True', 'apply_obstacle_ms': '0.0', 'candidate_state_valid': 'True', 'detection_check_ms': '2.007', 'detection_response_ms': '2.007', 'planning_status': 'planning_succeeded', 'moveit_error_code': '1', 'planning_response_ms': '305.777', 'plan_available_response_ms': '305.777', 'response_time_ms': '2.007', 'validation_ms': '108.795', 'trajectory_valid': 'True', 'checked_waypoints': '56', 'invalid_waypoints': '0', 'post_clear_scene_ok': 'True', 'post_clear_scene_ms': '0.0', 'success': 'True', 'failure_reason': 'none'}
{'trial_id': '2', 'target_name': 'Pose